In [ ]:
#@title 1. Google Drive se Connect Karein
from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive successfully connected!")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive successfully connected!


In [ ]:
!pip install PyMuPDF pillow pytesseract transformers pandas
!apt-get update
!apt-get install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-mal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 63.4 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,628 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [81.0 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,014 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu 

In [ ]:
import os, io, csv, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import fitz
from PIL import Image
import pytesseract
import cv2
from transformers import AutoTokenizer

ROOT_FOLDER   = "/content/drive/MyDrive/KMRL/MalayalamDocs"
OUTPUT_CSV    = "/content/drive/MyDrive/Malayalam1_extracted_token_new_chunks.csv"
pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"
TOKENIZER_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL, use_fast=True)
CHUNK_TOKENS, STRIDE_TOKENS = 256, 40


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
import os
import csv
import re
import unicodedata
import io
import numpy as np
import cv2
import pytesseract
from PIL import Image
from transformers import AutoTokenizer

# Assuming tokenizer is initialized outside these functions:
# tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", use_fast=True)

def unicode_has_malayalam(text):
    """Detects presence of Malayalam Unicode characters."""
    return bool(re.search(r'[\u0D00-\u0D7F]', text))

def ocr_page_better(page):
    """OCR fallback with image preprocessing to improve Malayalam+English extraction."""
    pix = page.get_pixmap(dpi=300)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    img_np = np.array(img)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    proc_img = Image.fromarray(thresh)
    return pytesseract.image_to_string(proc_img, lang='mal+eng').strip()

def robust_extract_text(page):
    """
    Try PyMuPDF text extraction. If text is too short or lacks Malayalam or English alphabets,
    fallback to OCR extraction.
    """
    raw = page.get_text("text")
    if (len(raw.strip()) > 20) and (re.search(r'[a-zA-Z]', raw) or unicode_has_malayalam(raw)):
        return raw
    return ocr_page_better(page)

def clean_text(text):
    """
    Normalize Unicode to NFKC form, remove unwanted control characters except newline/tab/space,
    and collapse multiple whitespaces to a single space.
    Works safely for both Malayalam and English.
    """
    if not isinstance(text, str):
        return ''
    text = unicodedata.normalize("NFKC", text)
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C' or ch in ('\n', '\t', ' '))
    return re.sub(r'\s+', ' ', text).strip()

def chunk_tokenwise(text, max_tokens=256, stride=40):
    """
    Tokenize the text using a multilingual tokenizer.
    Return overlapping chunks of tokens decoded back into text.
    """
    if not text:
        return []
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    n = len(token_ids)
    chunks = []
    i = 0
    chunk_id = 0
    while i < n:
        chunk_ids = token_ids[i:i+max_tokens]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        if chunk_text.strip():
            chunks.append((chunk_id, chunk_text, i, i + len(chunk_ids) - 1))
        chunk_id += 1
        if i + max_tokens >= n:
            break
        i += max_tokens - stride
    return chunks

def write_chunks_to_csv(chunks, output_csv, filename, department, page_number):
    """
    Append chunked text to a CSV file.
    CSV columns = ['source_file', 'department', 'page', 'chunk_index', 'text', 'token_start', 'token_end']
    """
    file_exists = os.path.exists(output_csv)
    with open(output_csv, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['source_file','department','page','chunk_index','text','token_start','token_end'])
        if not file_exists:
            writer.writeheader()
        for chunk_id, chunk_text, token_start, token_end in chunks:
            writer.writerow({
                'source_file': filename,
                'department': department,
                'page': page_number,
                'chunk_index': chunk_id,
                'text': chunk_text,
                'token_start': token_start,
                'token_end': token_end
            })


In [ ]:
def process_single_pdf(pdf_path, department, output_csv):
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"  - Error opening: {pdf_path}: {e}")
        return
    for page in doc:
        page_no = page.number + 1
        text = robust_extract_text(page)
        cleaned = clean_text(text)
        if not cleaned: continue
        chunks = chunk_tokenwise(cleaned)
        if chunks:
            write_chunks_to_csv(chunks, output_csv, os.path.basename(pdf_path), department, page_no)
            print(f"  - Page {page_no}: {len(chunks)} chunks.")
    doc.close()

def process_all_pdfs(root_folder, output_csv):
    if os.path.exists(output_csv): os.remove(output_csv)
    for dept_folder in Path(root_folder).iterdir():
        if dept_folder.is_dir():
            department = dept_folder.name
            print(f"\n📁 Dept: {department}")
            for pdf_path in dept_folder.glob("*.pdf"):
                print(f"  - {pdf_path.name}")
                process_single_pdf(pdf_path, department, output_csv)

process_all_pdfs(ROOT_FOLDER, OUTPUT_CSV)
print("\n✅ Extraction Complete!")



📁 Dept: ENGINEERING_Malayalam
  - SIA_Malayalam_KVHS-Palarivattom-to-Kakanad.pdf


Token indices sequence length is longer than the specified maximum sequence length for this model (1066 > 512). Running this sequence through the model will result in indexing errors


  - Page 1: 1 chunks.
  - Page 2: 5 chunks.
  - Page 3: 6 chunks.
  - Page 4: 5 chunks.
  - Page 5: 10 chunks.
  - Page 6: 12 chunks.
  - Page 7: 10 chunks.
  - Page 8: 9 chunks.
  - Page 9: 11 chunks.
  - Page 10: 7 chunks.
  - Page 11: 8 chunks.
  - Page 12: 13 chunks.
  - Page 13: 2 chunks.
  - Page 14: 12 chunks.
  - Page 15: 8 chunks.
  - Page 16: 7 chunks.
  - Page 17: 3 chunks.
  - Page 18: 6 chunks.
  - Page 19: 9 chunks.
  - Page 20: 7 chunks.
  - Page 21: 7 chunks.
  - Page 22: 6 chunks.
  - Page 23: 7 chunks.
  - Page 24: 9 chunks.
  - Page 25: 8 chunks.
  - Page 26: 11 chunks.
  - Page 27: 2 chunks.
  - Page 28: 8 chunks.
  - Page 29: 4 chunks.
  - Page 30: 7 chunks.
  - Page 31: 2 chunks.
  - Page 32: 2 chunks.
  - Page 33: 2 chunks.
  - Page 34: 2 chunks.
  - Page 35: 10 chunks.
  - Page 36: 6 chunks.
  - Page 37: 6 chunks.
  - Page 38: 6 chunks.
  - Page 39: 5 chunks.
  - Page 40: 5 chunks.
  - Page 41: 1 chunks.
  - Page 42: 2 chunks.
  - Page 43: 1 chunks.
  - Page 44:

KeyboardInterrupt: 